# 02. Dataset Audit — TruthLens UA Analytics

Ноутбук виконує аудит доступних даних для TruthLens UA Analytics. Основний локальний ресурс — `data/gold/demo_cases.csv`. Додатково робиться спроба завантажити відкритий датасет `lasr-unlp/unlp-2025-shared-task` з HuggingFace, якщо середовище підтримує таке завантаження.

In [1]:
from pathlib import Path
import pandas as pd
import re

try:
    from langdetect import detect
except Exception:
    detect = None

CSV_PATH = Path('../data/gold/demo_cases.csv')
df = pd.read_csv(CSV_PATH)
print(f'Завантажено {len(df)} кейсів із {CSV_PATH}')
df.head()

Завантажено 31 кейсів із ..\data\gold\demo_cases.csv


,id,text,expected_label,language,topic,ipso,explanation
0,1,ТЕРМІНОВО!!! ЗСУ ЗДАЛИ Харків! Поширте до вида...,FAKE,uk,military,"urgency_injection,military_disinfo,deletion_th...",3 ІПСО техніки: urgency+military_disinfo+delet...
1,2,ПРОКИНЬТЕСЬ! Приховують правду про вакцини! По...,FAKE,uk,health,"awakening_appeal,conspiracy_framing,viral_call",Змовницький наратив + заклик до поширення
2,3,Зеленський таємно підписав угоду з Путіним! Ан...,FAKE,uk,politics,"authority_impersonation,anonymous_sources",Імітація авторитету + анонімні джерела
3,4,ЗСУ ЗРАДНИКИ! КИНУЛИ ПОЗИЦІЇ! ПРАВДА ЯКУ ЗАМОВ...,FAKE,uk,military,"military_disinfo,conspiracy_framing,caps_abuse",Капслок + воєнна дезінформація + змова
4,5,Відео з генералом виявилось deepfake! Це AI-ві...,FAKE,uk,media,deepfake_indicator,Deepfake індикатор


In [2]:
def safe_detect(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return 'unknown'
    if detect is None:
        has_cyr = bool(re.search(r'[А-Яа-яІіЇїЄєҐґ]', text))
        return 'uk' if has_cyr else 'unknown'
    try:
        return detect(text)
    except Exception:
        return 'unknown'

language_counts = df['text'].astype(str).apply(safe_detect).value_counts().rename_axis('language').reset_index(name='count')
class_balance = df['expected_label'].value_counts().rename_axis('label').reset_index(name='count')
duplicate_count = int(df.duplicated(subset=['text']).sum())

print(f'Кількість дублікатів: {duplicate_count}')
class_balance, language_counts

Кількість дублікатів: 0


(        label  count
 0        REAL     15
 1        FAKE     10
 2  SUSPICIOUS      6,
   language  count
 0       uk     30
 1       ru      1)

In [3]:
hf_status = 'not_attempted'
hf_rows = 0
try:
    from datasets import load_dataset
    hf_ds = load_dataset('lasr-unlp/unlp-2025-shared-task', split='train', trust_remote_code=True)
    hf_rows = len(hf_ds)
    hf_status = 'loaded'
except Exception as exc:
    hf_status = f'unavailable: {type(exc).__name__}'

summary_table = pd.DataFrame([
    {'dataset': 'demo_cases.csv', 'rows': int(len(df)), 'language': 'uk', 'format': 'CSV', 'status': 'available'},
    {'dataset': 'lasr-unlp/unlp-2025-shared-task', 'rows': int(hf_rows), 'language': 'uk', 'format': 'HF dataset', 'status': hf_status}
])
summary_table

,dataset,rows,language,format,status
0,demo_cases.csv,31,uk,CSV,available
1,lasr-unlp/unlp-2025-shared-task,0,uk,HF dataset,unavailable: ModuleNotFoundError


In [4]:
datasets_md = f'''# Datasets — TruthLens UA Analytics\n\n| Датасет | Розмір | Мова | Формат | Статус |\n|---------|--------|------|--------|--------|\n| gold/demo_cases.csv | {len(df)} | uk | CSV | ✅ локально |\n| lasr-unlp/unlp-2025-shared-task | {hf_rows} | uk | HuggingFace | {hf_status} |\n\n## Audit summary\n- Кількість кейсів: {len(df)}\n- Кількість дублікатів: {duplicate_count}\n- Мовний розподіл: {language_counts.to_dict(orient='records')}\n- Баланс класів: {class_balance.to_dict(orient='records')}\n'''
output_path = Path('../docs/thesis/DATASETS.md')
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(datasets_md, encoding='utf-8')
print(datasets_md)
print(f'Файл збережено: {output_path}')

# Datasets — TruthLens UA Analytics

| Датасет | Розмір | Мова | Формат | Статус |
|---------|--------|------|--------|--------|
| gold/demo_cases.csv | 31 | uk | CSV | ✅ локально |
| lasr-unlp/unlp-2025-shared-task | 0 | uk | HuggingFace | unavailable: ModuleNotFoundError |

## Audit summary
- Кількість кейсів: 31
- Кількість дублікатів: 0
- Мовний розподіл: [{'language': 'uk', 'count': 30}, {'language': 'ru', 'count': 1}]
- Баланс класів: [{'label': 'REAL', 'count': 15}, {'label': 'FAKE', 'count': 10}, {'label': 'SUSPICIOUS', 'count': 6}]

Файл збережено: ..\docs\thesis\DATASETS.md
